In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.svm import SVC

In [4]:
df=pd.read_csv("../Datasets/Teen_Mental_Health_Dataset.csv")
df

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,18,female,6.8,Instagram,6.6,2.0,2.76,1.0,low,3,4,4,0
1196,16,male,2.3,Both,8.0,1.9,2.12,0.4,high,7,4,4,0
1197,14,female,1.7,Both,8.7,0.7,3.98,0.8,high,1,1,1,0
1198,15,male,3.9,Both,8.5,2.1,3.19,0.6,high,7,9,9,0


In [9]:
df.isnull().sum()

age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
dtype: int64

In [11]:
df_encoded=pd.get_dummies(df)
df_encoded

,age,daily_social_media_hours,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,stress_level,anxiety_level,addiction_level,depression_label,gender_female,gender_male,platform_usage_Both,platform_usage_Instagram,platform_usage_TikTok,social_interaction_level_high,social_interaction_level_low,social_interaction_level_medium
0,14,7.9,7.4,2.9,3.01,1.5,2,2,1,0,False,True,False,True,False,False,True,False
1,19,1.9,8.0,2.9,3.22,0.8,8,1,10,0,True,False,False,False,True,True,False,False
2,17,1.3,7.6,0.5,3.92,0.0,2,4,2,0,True,False,False,True,False,True,False,False
3,15,7.4,6.9,1.6,3.48,0.8,1,7,9,0,False,True,False,False,True,False,False,True
4,15,4.7,4.9,3.0,2.37,1.4,3,5,2,0,True,False,True,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,18,6.8,6.6,2.0,2.76,1.0,3,4,4,0,True,False,False,True,False,False,True,False
1196,16,2.3,8.0,1.9,2.12,0.4,7,4,4,0,False,True,True,False,False,True,False,False
1197,14,1.7,8.7,0.7,3.98,0.8,1,1,1,0,True,False,True,False,False,True,False,False
1198,15,3.9,8.5,2.1,3.19,0.6,7,9,9,0,False,True,True,False,False,True,False,False


In [47]:
test_sizes=[0.05, 0.10, 0.15, 0.20, 0.25]
models=[KNeighborsClassifier(n_neighbors=29),DecisionTreeClassifier(criterion = 'entropy',max_depth = 4,random_state = 42),RandomForestClassifier(n_estimators = 100,max_depth = 4,random_state = 42),SVC(C = 10, kernel = 'rbf', random_state = 42)]
score={}
for test_size in test_sizes:
    x=df_encoded.drop("depression_label",axis=1)
    y=df_encoded["depression_label"]
    x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=test_size,random_state=42)
    for model in models:
        model.fit(x_train,y_train)
        y_pred=model.predict(x_test)
        acc=accuracy_score(y_test,y_pred)
        data=score.get(model,[])
        data.append(acc)
        score[model]=data
score

{KNeighborsClassifier(n_neighbors=29): [0.9333333333333333,
  0.9583333333333334,
  0.9722222222222222,
  0.975,
  0.9733333333333334],
 DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=42): [1.0,
  1.0,
  1.0,
  1.0,
  0.9966666666666667],
 RandomForestClassifier(max_depth=4, random_state=42): [0.9333333333333333,
  0.9583333333333334,
  0.9722222222222222,
  0.975,
  0.9733333333333334],
 SVC(C=10, random_state=42): [0.95,
  0.975,
  0.9833333333333333,
  0.9833333333333333,
  0.9833333333333333]}

In [48]:
score_df=pd.DataFrame(score)
score_df.columns=["KNN","TREE","RF","SVM"]
score_df

,KNN,TREE,RF,SVM
0,0.933333,1.000000,0.933333,0.950000
1,0.958333,1.000000,0.958333,0.975000
2,0.972222,1.000000,0.972222,0.983333
3,0.975000,1.000000,0.975000,0.983333
4,0.973333,0.996667,0.973333,0.983333


In [51]:
score_df["test_size"]=test_sizes

In [52]:
score_df

,KNN,TREE,RF,SVM,test_size
0,0.933333,1.000000,0.933333,0.950000,0.05
1,0.958333,1.000000,0.958333,0.975000,0.10
2,0.972222,1.000000,0.972222,0.983333,0.15
3,0.975000,1.000000,0.975000,0.983333,0.20
4,0.973333,0.996667,0.973333,0.983333,0.25


In [54]:
px.line(score_df,x="test_size",y=[score_df.KNN,score_df.TREE,score_df.RF,score_df.SVM])

In [60]:
max_accuracy={"KNN":score_df.KNN.max(),"TREE":score_df.TREE.max(),"RF":score_df.RF.max(),"SVM":score_df.SVM.max()}
max_accuracy
px.bar(x=max_accuracy.keys(),y=max_accuracy.values())